# d2_eval — Direction 2: MPC in the organisers' simulator, against K10

Runs each policy for `STEPS` steps (`hvac_controller/d1/evaluate.py`), one fresh process per scenario × policy × seed,
and compares it per scenario with K10 (the public 0.0185 baseline).

- `POLICIES`: comma-separated. `k10`; `d2:meta` (the submission: model rebuilt from the metadata);
  `d2:oracle` (diagnostic only: plans on the scenario's own replica cache, which `data` builds here).
  MPC overrides go after dots, e.g. `d2:meta.rp24.n32.it2` (`hvac_controller/d2/mpc.py`, `parse_config`).
- `SCENARIOS`: `"dev"` (10: one strip per station across buildings, plus two on the unused `minneapolis` station),
  `"shipped"` (the 30), or a comma-separated `building/weather` list.
- `WHERE`: `original` (the organisers' simulator, the truth) or `replica` (the exact replica of d1, about 10x
  faster; scores agree with the original to the digits shown on the shipped scenarios). `BASE`: the reference policy.
- `STEPS`: 8640 is the scored length. Never compare runs of different lengths.
- Jobs are dealt to `N_SHARDS` sessions by `RUN_ID` (two Save Versions run at once).

**Inputs:** the competition data. CPU, **Internet on** (clones the repo).

**Output:** `/kaggle/working/d2_eval_<RUN_ID>.zip`: one JSON per run; the comparison is printed at the end.

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction2"
POLICIES = "k10,d2:meta.rp24.lag6.it2"
SCENARIOS = "dev"          # "dev" (10), "shipped" (30) or "b/w,b/w,..."
STEPS = 8640
WHERE = "original"         # "original" (the organisers' simulator) or "replica" (the exact replica, ~10x faster)
BASE = "k10"               # policy the others are compared with
SEEDS = 1
RUN_ID, N_SHARDS = 0, 2
N_PROCS = 4

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

REPO, INPUT, WORK = Path("/tmp/hvac-controller"), Path("/kaggle/input"), Path("/tmp/work")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
sys.path.insert(0, str(REPO))
# One BLAS thread per process: N_PROCS processes with multi-threaded BLAS each fight over the 4 CPUs.
ENV = dict(os.environ, PYTHONPATH=str(REPO), OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")

SHIPPED = next(p for p in sorted(INPUT.rglob("training_scenarios"))
               if (p / "building_train_1" / "atlanta_2" / "config.gin").exists())
ROOT = REPO / "smart_buildings" / "training_scenarios"
shutil.copytree(SHIPPED, ROOT, dirs_exist_ok=True)

DEV = ["building_train_1/atlanta_2", "building_train_2/phoenix_6", "building_train_4/fairbanks_4",
       "building_train_5/denver_3", "building_train_7/singapore_6", "building_train_8/fairbanks_8",
       "building_train_8/phoenix_4", "building_train_2/denver_8",
       "building_train_4/minneapolis_3", "building_train_7/minneapolis_6"]  # minneapolis: no shipped scenario uses it
if SCENARIOS == "dev":
    LIST = DEV
elif SCENARIOS == "shipped":
    LIST = sorted(f"{p.parent.parent.name}/{p.parent.name}" for p in ROOT.glob("*/*/config.gin"))
else:
    LIST = SCENARIOS.split(",")
for s in LIST:  # build any scenario that does not ship (the generator rebuilds shipped ones exactly)
    b, w = s.split("/")
    if not (ROOT / b / w / "config.gin").exists():
        r = subprocess.run([sys.executable, "-m", "smart_buildings.run_scenario_generator", "--buildings", b,
                            "--weathers", w], cwd=REPO, env=ENV, capture_output=True, text=True)
        print("generated", s, r.returncode, (r.stdout + r.stderr)[-300:])
HERE = LIST[RUN_ID::N_SHARDS]
print(f"{len(HERE)} of {len(LIST)} scenarios here:", HERE)

In [ ]:
# Replica runs and oracle policies need the scenario's replica cache (d1/cache.py): ~30 s to minutes each.
CACHE = WORK / "cache"
if WHERE == "replica" or any(k in POLICIES for k in ("d2:oracle", "d2:lpo", "d2:heavy")):
    for s in HERE:
        out = CACHE / f"{s.replace('/', '__')}.pkl"
        if not out.exists():
            r = subprocess.run([sys.executable, "-m", "hvac_controller.d1.cache", str(out), "--config",
                                str(ROOT / s / "config.gin")], cwd=REPO, env=ENV, capture_output=True, text=True)
            print((r.stdout + r.stderr).strip()[-300:])

In [ ]:
# The runs, N_PROCS at a time; each result is printed as it ends.
EVAL = WORK / "eval"
cmd = [sys.executable, "-m", "hvac_controller.d1.evaluate", str(EVAL), "--scenarios", ",".join(HERE),
       "--scenario-root", str(ROOT), "--cache-dir", str(CACHE), "--policies", POLICIES, "--seeds", str(SEEDS),
       "--steps", str(STEPS), "--n-procs", str(N_PROCS), "--where", WHERE]
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=REPO, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if line.startswith("[") or "original" in line or "Error" in line:
        print(line, end="", flush=True)
print("exit", proc.wait(), f"after {(time.time() - t0) / 60:.0f} min")

In [ ]:
# Summary: per scenario score, terms and timing; candidate vs K10; the zip.
from hvac_controller.d1 import evaluate as ev

results = [json.loads(p.read_text()) for p in sorted(EVAL.glob("*.json"))]
for r in results:
    if "error" in r:
        print("ERROR", r["scenario"], r["policy"], r["error"], r["traceback"][-1500:])
print(f"{'scenario':30s} {'policy':22s} {'score':>8s} {'regret':>8s} {'cost':>7s} {'carbon':>7s} {'pol ms':>7s} {'min':>5s}")
for r in sorted(results, key=lambda r: (r["scenario"], r["policy"])):
    if "error" not in r:
        print(f"{r['scenario']:30s} {ev.policy_tag(r['policy']):22s} {r['score']:8.5f} {r['regret']:8.4f} {r['cost']:7.4f} "
              f"{r['carbon']:7.4f} {r['policy_ms']:7.1f} {r['seconds'] / 60:5.1f}  {r.get('mpc', '')}")
ens = ev.ensembles(results)
for cand in [ev.policy_tag(p) for p in POLICIES.split(",") if p != BASE]:
    cmp = ev.compare(ens, cand, ev.policy_tag(BASE), WHERE)
    if cmp:
        m = cmp.pop("mean")
        print(f"\n{cand} vs {BASE} ({WHERE}) on {len(cmp)} scenarios: mean {m['candidate']:.5f} vs {m['null']:.5f} "
              f"({m['candidate'] / m['null'] - 1:+.1%}); better on {sum(c['rel'] < 0 for c in cmp.values())}")
name = Path(f"/kaggle/working/d2_eval_{RUN_ID}.zip")
with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(EVAL.glob("*.json")):
        z.write(p, p.name)
print("wrote", name)